In [ ]:
# Practice Problems Day 13

### Author: Joanna Bieri (solutions)

## Introduction to Machine Learning

#### University of Redlands - DATA 301
#### Prof: Joanna Bieri [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
#### Day 13 Practice - SOLUTIONS
#### [Class Website](https://joannabieri.com/machine_learning.html)

---

**These are practice problems. They are not graded and you do not turn them in.** They are here so you can practice today's code before the weekly homework. Weekly Homework 7 assumes you have done them. Plan on about 45 minutes.

**These are my solutions.** Try every problem yourself first, then come here to check your work. There is more than one right way to write the code, and your words will be different from mine, that is fine! If your numbers do not match, look for the difference before you look at my code.

**Reading:** Geron chapter 11, **Avoiding Overfitting Through Regularization** (Dropout, and ℓ₁ and ℓ₂ Regularization), and chapter 4, **Early Stopping**.

GOALS:

1. Spot overfitting in the training and validation RMSE.
2. Add dropout to a network.
3. Add early stopping to a training loop.

**How much of this should you write yourself?** The starting cell is given. Problem 1 adds a few lines to it. Problem 2 lists the steps. Problem 3 is words only. The **Optional exploration** at the end is extra, only if you are curious. Each training run takes less than a minute on a laptop.

**Save your work.** Commit and push when you are done, so you have your code for the weekly homework and the exam:

```bash
git add .
git commit -m "Day 13 practice"
git push
```

---

In [ ]:
# RUN THIS. The California housing data from the notes, with only the first 2000 training rows.
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

housing = pd.read_csv("data/california_housing.csv")
X = housing.drop(columns=["MedHouseVal"])
y = housing["MedHouseVal"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=42)
X_train = X_train[:2000]                                       # only the first 2000 training rows, like the notes
y_train = y_train[:2000]

preprocessing = Pipeline([("scale", StandardScaler())])      # the preprocessing pipeline, fit on the training rows only
X_train_scaled = preprocessing.fit_transform(X_train)
X_valid_scaled = preprocessing.transform(X_valid)

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
X_valid_t = torch.tensor(X_valid_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
y_valid_t = torch.tensor(y_valid.values, dtype=torch.float32).reshape(-1, 1)
train_data = TensorDataset(X_train_t, y_train_t)               # pairs the features with the labels for the DataLoader

print("training rows:", X_train_t.shape[0], "   validation rows:", X_valid_t.shape[0])

In [ ]:
# RUN THIS. A network with 3 hidden layers of 100 neurons and no regularization, trained for 100 epochs.
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(8, 100),                        # 8 features in, 100 neurons
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 1))                        # 1 prediction out

# the starting weights
for layer in model:
    if isinstance(layer, nn.Linear):          # only the Linear layers have weights
        nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
        nn.init.zeros_(layer.bias)

# the batches, the loss and the optimizer
torch.manual_seed(42)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# the training loop
for epoch in range(100):
    model.train()                             # training mode
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
        loss.backward()
        optimizer.step()

    # every 20 epochs, print the training and validation RMSE
    if (epoch + 1) % 20 == 0:
        model.eval()                          # prediction mode
        with torch.no_grad():
            train_rmse = torch.sqrt(loss_fn(model(X_train_t), y_train_t)).item()
            valid_rmse = torch.sqrt(loss_fn(model(X_valid_t), y_valid_t)).item()
        print("epoch", epoch + 1, "  train RMSE:", round(train_rmse, 4), "  valid RMSE:", round(valid_rmse, 4))

## Problem 1: Spot the overfitting, then add dropout

**What to copy, what to write:** The cell above is given. Copy it into the cell below and add the dropout lines yourself.

**1a.** Run the starting cell. Is this network overfitting? Point at the numbers that tell you.

**1b.** Copy the cell and add `nn.Dropout(p=0.1)` after each `nn.ReLU()`. Run it. Compare the training RMSE and the validation RMSE after 100 epochs to 1a. What changed?

In [ ]:
# 1b: the same cell with dropout after each ReLU
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(8, 100),                        # 8 features in, 100 neurons
    nn.ReLU(),
    nn.Dropout(p=0.1),                    # dropout after the activation
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Dropout(p=0.1),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Dropout(p=0.1),
    nn.Linear(100, 1))                        # 1 prediction out

# the starting weights
for layer in model:
    if isinstance(layer, nn.Linear):          # only the Linear layers have weights
        nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
        nn.init.zeros_(layer.bias)

# the batches, the loss and the optimizer
torch.manual_seed(42)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# the training loop
for epoch in range(100):
    model.train()                             # training mode
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
        loss.backward()
        optimizer.step()

    # every 20 epochs, print the training and validation RMSE
    if (epoch + 1) % 20 == 0:
        model.eval()                          # prediction mode
        with torch.no_grad():
            train_rmse = torch.sqrt(loss_fn(model(X_train_t), y_train_t)).item()
            valid_rmse = torch.sqrt(loss_fn(model(X_valid_t), y_valid_t)).item()
        print("epoch", epoch + 1, "  train RMSE:", round(train_rmse, 4), "  valid RMSE:", round(valid_rmse, 4))

**1a.** Yes. The training RMSE keeps going down, to 0.2698 after 100 epochs, but the validation RMSE does not follow it: it is 0.6135 at epoch 20 and 0.6353 at epoch 100. The gap between them keeps growing. The network is memorizing the 2000 training rows.

**1b.** With dropout, after 100 epochs the training RMSE is 0.4343 and the validation RMSE is 0.5835. The training RMSE got **worse**, and that is good: dropout makes it hard to memorize the training rows. The two numbers are much closer together now, and the validation RMSE is better too.

---

## Problem 2: Add early stopping (steps outlined)

**What to copy, what to write:** Start from your dropout cell from 1b. Write the early stopping lines yourself, following the "real life" cell at the end of the notes.

**2a.** Add early stopping with a patience of **15**, and let it train for at most 200 epochs. Print the epoch where it stopped, the best epoch, and the best validation RMSE.

**2b.** Is the best validation RMSE better than what you got after 100 epochs in 1b? How many epochs did the network train this time?

In [ ]:
# 2a: dropout and early stopping, patience 15
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(8, 100),
    nn.ReLU(),
    nn.Dropout(p=0.1),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Dropout(p=0.1),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Dropout(p=0.1),
    nn.Linear(100, 1))

for layer in model:
    if isinstance(layer, nn.Linear):
        nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
        nn.init.zeros_(layer.bias)

torch.manual_seed(42)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# early stopping
patience = 15                                # how many epochs in a row we wait for a new best
best_rmse = float("inf")                      # the best validation RMSE so far
best_weights = None
epochs_without_improving = 0

for epoch in range(200):                      # the most epochs we are willing to train
    model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
        loss.backward()
        optimizer.step()

    model.eval()                              # check the validation RMSE after EVERY epoch
    with torch.no_grad():
        rmse = torch.sqrt(loss_fn(model(X_valid_t), y_valid_t)).item()

    if rmse < best_rmse:                      # a new best, save a copy of the weights
        best_rmse = rmse
        best_epoch = epoch + 1
        best_weights = copy.deepcopy(model.state_dict())
        epochs_without_improving = 0
    else:
        epochs_without_improving = epochs_without_improving + 1

    if epochs_without_improving == patience:
        print("stopped after epoch", epoch + 1)
        break

model.load_state_dict(best_weights)           # put the best weights back
print("best epoch:", best_epoch, "  validation RMSE:", round(best_rmse, 4))

**2a.** It stopped after epoch 73. The best epoch was 58, with a validation RMSE of 0.5769.

**2b.** Yes, a little: 0.5769 against the 0.5835 we had after 100 epochs in 1b. And it only trained for 73 epochs instead of 100. Early stopping does not stop at a fixed number of epochs, it stops when the validation RMSE has not improved for 15 epochs in a row, and then it puts back the weights from the best epoch.

---

## Problem 3: In words

**3a.** Explain **early stopping** to a friend in DATA 101 who knows what overfitting is but has never trained a neural network. Three or four sentences, no code.

**3b.** A classmate's network has a training RMSE of 0.05 and a validation RMSE of 0.9. Name two things from today they could try, and say why each one could help.

**3a.** Your answer will be different, here is one way to say it:

While a neural network trains, it goes through the training data over and over. At first it gets better on both the training data and new data. After a while it starts memorizing the training data, and it gets worse on new data. Early stopping checks the error on a validation set after every pass, keeps a copy of the model from the best pass, and stops once the validation error has not improved for a while.

**3b.** Two good things to try:

1. **Early stopping.** Keep the weights from the epoch with the best validation RMSE, before the network memorized the training rows.
2. **Dropout.** Turning off random neurons while training makes it hard for the network to memorize, so the training and validation RMSE stay closer together.

More training data, or weight decay, are also fair answers.

---

## Optional exploration

Only if you are curious. Nothing here is needed for the weekly homework.

**A.** In your early stopping cell, take out the dropout layers and add `weight_decay=0.0001` to the optimizer instead. How does it compare?

**B.** Change the dropout rate to 0.5. Is more dropout better here?

In [ ]:
# Optional A: weight decay instead of dropout
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(8, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 1))

for layer in model:
    if isinstance(layer, nn.Linear):
        nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
        nn.init.zeros_(layer.bias)

torch.manual_seed(42)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)

# early stopping
patience = 15                                # how many epochs in a row we wait for a new best
best_rmse = float("inf")                      # the best validation RMSE so far
best_weights = None
epochs_without_improving = 0

for epoch in range(200):                      # the most epochs we are willing to train
    model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
        loss.backward()
        optimizer.step()

    model.eval()                              # check the validation RMSE after EVERY epoch
    with torch.no_grad():
        rmse = torch.sqrt(loss_fn(model(X_valid_t), y_valid_t)).item()

    if rmse < best_rmse:                      # a new best, save a copy of the weights
        best_rmse = rmse
        best_epoch = epoch + 1
        best_weights = copy.deepcopy(model.state_dict())
        epochs_without_improving = 0
    else:
        epochs_without_improving = epochs_without_improving + 1

    if epochs_without_improving == patience:
        print("stopped after epoch", epoch + 1)
        break

model.load_state_dict(best_weights)           # put the best weights back
print("best epoch:", best_epoch, "  validation RMSE:", round(best_rmse, 4))

In [ ]:
# Optional B: a dropout rate of 0.5
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(8, 100),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(100, 1))

for layer in model:
    if isinstance(layer, nn.Linear):
        nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
        nn.init.zeros_(layer.bias)

torch.manual_seed(42)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# early stopping
patience = 15                                # how many epochs in a row we wait for a new best
best_rmse = float("inf")                      # the best validation RMSE so far
best_weights = None
epochs_without_improving = 0

for epoch in range(200):                      # the most epochs we are willing to train
    model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
        loss.backward()
        optimizer.step()

    model.eval()                              # check the validation RMSE after EVERY epoch
    with torch.no_grad():
        rmse = torch.sqrt(loss_fn(model(X_valid_t), y_valid_t)).item()

    if rmse < best_rmse:                      # a new best, save a copy of the weights
        best_rmse = rmse
        best_epoch = epoch + 1
        best_weights = copy.deepcopy(model.state_dict())
        epochs_without_improving = 0
    else:
        epochs_without_improving = epochs_without_improving + 1

    if epochs_without_improving == patience:
        print("stopped after epoch", epoch + 1)
        break

model.load_state_dict(best_weights)           # put the best weights back
print("best epoch:", best_epoch, "  validation RMSE:", round(best_rmse, 4))

**A.** With weight decay and no dropout, early stopping stopped after epoch 35, with a best validation RMSE of 0.5977 at epoch 20. That is a little worse than dropout with early stopping here (0.5769).

**B.** No. With a dropout rate of 0.5 the best validation RMSE was 0.7121 (epoch 62), much worse than 0.5769 with 0.1. Turning off half of the neurons at every step makes it too hard for this network to learn anything, so too much dropout hurts too.